In [ ]:
import pandas as pd

def clean_gold(df):
    # Drop columns: 'Unnamed: 0', 'Unnamed: 1' and 66 other columns
    df = df.drop(columns=['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 16', 'Unnamed: 15', 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24', 'Unnamed: 25', 'Unnamed: 26', 'Unnamed: 27', 'Unnamed: 29', 'Unnamed: 28', 'Unnamed: 30', 'Unnamed: 31', 'Unnamed: 32', 'Unnamed: 33', 'Unnamed: 34', 'Unnamed: 35', 'Unnamed: 36', 'Unnamed: 37', 'Unnamed: 38', 'Unnamed: 39', 'Unnamed: 40', 'Unnamed: 41', 'Unnamed: 42', 'Unnamed: 43', 'Unnamed: 44', 'Unnamed: 45', 'Unnamed: 46', 'Unnamed: 47', 'Unnamed: 48', 'Unnamed: 50', 'Unnamed: 51', 'Unnamed: 52', 'Unnamed: 53', 'Unnamed: 54', 'Unnamed: 55', 'Unnamed: 57', 'Unnamed: 58', 'Unnamed: 59', 'Unnamed: 60', 'Unnamed: 61', 'Unnamed: 62', 'Unnamed: 63', 'Unnamed: 64', 'Unnamed: 65', 'Unnamed: 66', 'Unnamed: 68', 'Unnamed: 67', 'Unnamed: 69', 'Unnamed: 71', 'Unnamed: 70'])
    df = df.drop(df.index[:10])
    # Rename column 'Unnamed: 6' to 'Date'
    df = df.rename(columns={'Unnamed: 6': 'Date'})
    df = df.rename(columns={'Unnamed: 12': 'Order #'})
    df = df.rename(columns={'Unnamed: 49': 'Gold Debit'})
    df = df.rename(columns={'Unnamed: 56': 'Credit'})
    # Convert 'Credit' column to numeric and negate its absolute values
    df['Credit'] = df['Credit'].str.replace(',', '', regex=False)
    df['Credit'] = pd.to_numeric(df['Credit'], errors='coerce')
    df['Credit'] = -df['Credit'].abs()
    df['Gold Debit'] = df['Gold Debit'].str.replace(',', '', regex=False)
    df['Gold Debit'] = pd.to_numeric(df['Gold Debit'], errors='coerce')
    df['Credit'] = df['Credit'].replace(-0, 0)
    # Ensure 'Date' column is in datetime format
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
    # Format 'Date' column to desired string format
    df['Date'] = df['Date'].dt.strftime('%m/%d/%Y')
    return df

# Loaded variable 'df' from URI: c:\Users\MichaelGinsberg\Documents\TPM_Development\accounting_reconciling_match\Data\gold_ledger.csv
df = pd.read_csv(r'c:\Users\MichaelGinsberg\Documents\TPM_Development\accounting_reconciling_match\Data\gold_ledger.csv')

df_gold = clean_gold(df.copy())
df_gold.head()

,Date,Order #,Gold Debit,Credit
10,01/02/2026,XPO042611,21574.75,0.0
11,01/02/2026,XPO042611,4314.95,0.0
12,01/02/2026,XPO042611,17259.80,0.0
13,01/02/2026,XPO042735,2209625.00,0.0
14,01/02/2026,XPO042748,86672.40,0.0


In [ ]:
import pandas as pd

def clean_inv(df):
    # Drop columns: 'SITE', 'FROM SITE' and 9 other columns
    df = df.drop(columns=['SITE', 'FROM SITE', 'ACCOUNT', 'UC1', 'UC2', 'UC3', 'UC4', 'JOURNAL', 'Item', 'C/V NUM', 'Name'])
    # Rename column 'Document Number' to 'Order #'
    df = df.rename(columns={'Document Number': 'Order #'})
    # Rename column 'DATE' to 'Date'
    df = df.rename(columns={'DATE': 'Date'})
    # Fix column assignment to DataFrame
    df['Date'] = df['Date'].dt.strftime('%m/%d/%Y')
    # Drop column: 'REFERENCE'
    df = df.drop(columns=['REFERENCE'])
    # Rename column 'AMOUNT' to 'Credit'
    df = df.rename(columns={'AMOUNT': 'Credit'})
    df['Credit'] = pd.to_numeric(df['Credit'], errors='coerce')
    return df

# Loaded variable 'df' from URI: c:\Users\MichaelGinsberg\Documents\TPM_Development\accounting_reconciling_match\Data\Finance_ueGLDetailwItemDataViewExport4_16_2026.xlsx
df = pd.read_excel(r'c:\Users\MichaelGinsberg\Documents\TPM_Development\accounting_reconciling_match\Data\Finance_ueGLDetailwItemDataViewExport4_16_2026.xlsx')

df_inv = clean_inv(df.copy())
df_inv.head()

,Date,Credit,Order #
0,01/02/2026,-43606.40,XPO042752
1,01/02/2026,-50.00,XPO042752
2,01/02/2026,-1012.93,XPO042783
3,01/02/2026,20.00,XPO042783
4,01/02/2026,-4318.15,XPO042792


In [39]:
# Build lookup: (Date, Credit) -> Order # from df_inv
# Exclude rows with no Order # to avoid overwriting valid df_gold values with NaN
inv_lookup = (
    df_inv
    .dropna(subset=['Order #'])
    .drop_duplicates(subset=['Date', 'Credit'])
    .set_index(['Date', 'Credit'])['Order #']
    .to_dict()
)

# Replace Order # in df_gold if either:
#   - Date + Credit match df_inv's Date + Credit, OR
#   - Date + Gold Debit match df_inv's Date + Credit
# Falls back to original Order # if neither matches
df_gold['Order #'] = [
    inv_lookup.get((date, credit), inv_lookup.get((date, gold_debit), orig))
    for date, credit, gold_debit, orig in zip(
        df_gold['Date'], df_gold['Credit'], df_gold['Gold Debit'], df_gold['Order #']
    )
]

df_gold.head()

,Date,Order #,Gold Debit,Credit
10,01/02/2026,XPO042611,21574.75,0.0
11,01/02/2026,XPO042611,4314.95,0.0
12,01/02/2026,XPO042611,17259.80,0.0
13,01/02/2026,XPO042735,2209625.00,0.0
14,01/02/2026,XPO042748,86672.40,0.0


In [40]:
df_summary = (
    df_gold
    .groupby('Order #', as_index=False)[['Gold Debit', 'Credit']]
    .sum()
)
df_summary['Remainder'] = df_summary['Gold Debit'] + df_summary['Credit']
df_summary

,Order #,Gold Debit,Credit,Remainder
0,MODXPO 44734,405.00,-16107.38,-15702.38
1,XPO041427,4003.42,0.00,4003.42
2,XPO042361,49807.50,0.00,49807.50
3,XPO042368,81356.10,0.00,81356.10
4,XPO042384,280792.20,0.00,280792.20
...,...,...,...,...
353,XPO044753,1515.00,-64765.48,-63250.48
354,XPO044755,372112.50,-372112.50,0.00
355,XPO044756,24709.50,-24709.50,0.00
356,XPO044766,0.00,-770046.43,-770046.43


In [41]:
df_grouped = df_gold.groupby('Order #', as_index=False)[['Gold Debit', 'Credit']].sum()
df_grouped['Gold Debit'] = df_grouped['Gold Debit'].round(2)
df_grouped['Credit'] = df_grouped['Credit'].round(2)
df_grouped['Remainder'] = df_grouped['Gold Debit'] + df_grouped['Credit']
df_grouped

,Order #,Gold Debit,Credit,Remainder
0,MODXPO 44734,405.00,-16107.38,-15702.38
1,XPO041427,4003.42,0.00,4003.42
2,XPO042361,49807.50,0.00,49807.50
3,XPO042368,81356.10,0.00,81356.10
4,XPO042384,280792.20,0.00,280792.20
...,...,...,...,...
353,XPO044753,1515.00,-64765.48,-63250.48
354,XPO044755,372112.50,-372112.50,0.00
355,XPO044756,24709.50,-24709.50,0.00
356,XPO044766,0.00,-770046.43,-770046.43


In [ ]:
df_grouped.to_excel('gold_cleaned.xlsx', index=False)